# GEBCO global isobath extraction

This notebook reproduces the six basin-boundary functions from `extract_isobath.ipynb` using the GEBCO 2026 sub-ice bathymetry. The 15 arc-second source grid is block-averaged to 1 arc-minute before the original shelf filtering, bridge construction, row-wise extraction, and cleaning are applied at 500 m and 1000 m.

In [1]:
from pathlib import Path
import gc

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from dask.diagnostics import ProgressBar

In [2]:
REPO_ROOT = Path("/Users/awatford/Documents/GitHub/MOCAdjustmentTheory/")
GEBCO_PATH = REPO_ROOT / "data/untracked/GEBCO/GEBCO_2026_sub_ice/GEBCO_2026_sub_ice.nc"
ISOBATH_DIRECTORY = REPO_ROOT / "data/tracked/isobath/"

COARSEN_FACTOR = 4
TARGET_RESOLUTION_DEG = COARSEN_FACTOR / 240.0
TARGET_RESOLUTION_ARCMIN = TARGET_RESOLUTION_DEG * 60.0
WORK_LAT_BOUNDS = (-60.5, 72.5)
OUTPUT_LAT_BOUNDS = (-80.0, 90.0)

# Full-longitude chunks follow the contiguous row layout of the source file.
gebco_source = xr.open_dataset(GEBCO_PATH, chunks={"lat": 240, "lon": -1})
native_work = gebco_source.elevation.sel(lat=slice(*WORK_LAT_BOUNDS))
coarse_elevation = (
    native_work.coarsen(lat=COARSEN_FACTOR, lon=COARSEN_FACTOR, boundary="trim")
    .mean()
    .astype(np.float32)
)

with ProgressBar():
    coarse_elevation = coarse_elevation.compute()

# Convert GEBCO height (positive upward) to the positive ocean depth convention
# expected by the original extraction code, reusing the allocated float32 array.
depth_values = coarse_elevation.values
np.negative(depth_values, out=depth_values)
np.maximum(depth_values, 0.0, out=depth_values)

deptho = xr.DataArray(
    depth_values,
    dims=("latitude", "longitude"),
    coords={
        "latitude": coarse_elevation.lat.values.astype(float),
        "longitude": coarse_elevation.lon.values.astype(float),
    },
    name="deptho",
    attrs={"units": "m", "source": "GEBCO 2026 sub-ice grid"},
)

output_latitude = (
    gebco_source.lat.coarsen(lat=COARSEN_FACTOR, boundary="trim")
    .mean()
    .sel(lat=slice(*OUTPUT_LAT_BOUNDS))
    .values.astype(float)
)

# The original 2,500-cell component threshold is scaled by the square of the
# resolution ratio so it represents the same approximate geographic area.
GLORYS_RESOLUTION_DEG = 1.0 / 12.0
MIN_BARRIER_CELLS = int(round(2500 * (GLORYS_RESOLUTION_DEG / TARGET_RESOLUTION_DEG) ** 2))

print(f"GEBCO source: {gebco_source.elevation.shape} at 15 arc-seconds")
print(f"Working grid: {deptho.shape} at {TARGET_RESOLUTION_ARCMIN:g} arc-minute")
print(f"Output latitude points: {output_latitude.size}")
print(f"Scaled minimum barrier size: {MIN_BARRIER_CELLS:,} cells")

[                                        ] | 0% Completed | 96.96 us

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


[########################################] | 100% Completed | 1.46 sms
GEBCO source: (43200, 86400) at 15 arc-seconds
Working grid: (7980, 21600) at 1 arc-minute
Output latitude points: 10200
Scaled minimum barrier size: 62,500 cells


In [3]:
from scipy import ndimage
from skimage import graph, measure
import matplotlib.pyplot as plt

# Basin windows are expressed in a basin-specific unwrapped longitude coordinate.
# Pacific longitudes are 0..360 so its eastern boundary is North/South America.
# Step-shaped southern limits for each physical boundary in the schematic.
# Atlantic west and Pacific east start at Drake Passage / southern South America;
# Atlantic east and Indian west start at South Africa; Indian east and Pacific
# west start at southern Australia/Tasmania.
STEP_LATITUDES = {
    "south_america": -56.0,
    "south_africa": -35.0,
    "southern_australia": -44.0,
}

BOUNDARY_SOUTHERN_LIMITS = {
    "atlantic": {
        "west": STEP_LATITUDES["south_america"],
        "east": STEP_LATITUDES["south_africa"],
    },
    "indian": {
        "west": STEP_LATITUDES["south_africa"],
        "east": STEP_LATITUDES["southern_australia"],
    },
    "pacific": {
        "west": STEP_LATITUDES["southern_australia"],
        "east": STEP_LATITUDES["south_america"],
    },
}

# Northern cleanup limits trim Arctic/side-sea artifacts that otherwise appear
# as inland Greenland or China branches in the single-valued functions.
BOUNDARY_NORTHERN_LIMITS = {
    "atlantic": {"west": 65.0, "east": 65.0},
    "indian": {"west": 25.0, "east": 25.0},
    "pacific": {"west": 60.0, "east": 60.0},
}

BASIN_CONFIGS = {
    "atlantic": {
        "center": 0.0,
        "lon_bounds": (-100.0, 40.0),
        "lat_bounds": (min(BOUNDARY_SOUTHERN_LIMITS["atlantic"].values()), 72.0),
        "west_south": BOUNDARY_SOUTHERN_LIMITS["atlantic"]["west"],
        "east_south": BOUNDARY_SOUTHERN_LIMITS["atlantic"]["east"],
        "seed_lon": -30.0,
        "color": "tab:blue",
    },
    "indian": {
        "center": 75.0,
        "lon_bounds": (20.0, 150.0),
        "lat_bounds": (min(BOUNDARY_SOUTHERN_LIMITS["indian"].values()), 30.0),
        "west_south": BOUNDARY_SOUTHERN_LIMITS["indian"]["west"],
        "east_south": BOUNDARY_SOUTHERN_LIMITS["indian"]["east"],
        "seed_lon": 75.0,
        "color": "tab:orange",
    },
    "pacific": {
        "center": 180.0,
        "lon_bounds": (105.0, 290.0),
        "lat_bounds": (min(BOUNDARY_SOUTHERN_LIMITS["pacific"].values()), 66.0),
        "west_south": BOUNDARY_SOUTHERN_LIMITS["pacific"]["west"],
        "east_south": BOUNDARY_SOUTHERN_LIMITS["pacific"]["east"],
        "seed_lon": 200.0,
        "color": "tab:green",
    },
}

# Large isolated islands that should not define continental-shelf basin walls.
# These are deliberately removed before row-wise west/east boundary extraction.
EXCLUDED_ISLAND_BOXES = [
    ("madagascar", 42.0, 52.5, -27.0, -10.0),
    ("new_zealand", 165.0, 180.0, -49.5, -33.0),
]

# These are only final cleanup bridges. The boundary functions are extracted
# row-by-row from the bathymetry after small islands are ignored.
BRIDGES = [
    {
        "name": "greenland_iceland_uk_europe",
        "points": [(5.0, 60.0), (-6.5, 62.0), (-18.5, 64.8), (-38.0, 66.0)],
        "width_deg": 0.45,
        "method": "route",
        "pad_deg": 4.0,
    },
    {
        "name": "se_asia_to_australia",
        "points": [(103.0, 1.0), (110.0, -6.5), (121.0, -8.8), (130.0, -10.5), (142.0, -12.0)],
        "width_deg": 0.45,
        "method": "route",
        "pad_deg": 5.0,
    },
    {
        "name": "bering_strait",
        "points": [(165.0, 65.5), (179.0, 65.5), (-166.0, 65.5)],
        "width_deg": 0.55,
        "method": "line",
    },
    {
        "name": "caribbean_arc",
        "points": [(-81.5, 24.5), (-77.5, 21.5), (-73.0, 19.0), (-66.5, 18.0), (-61.5, 14.5)],
        "width_deg": 0.35,
        "method": "route",
        "pad_deg": 3.0,
    },
]


def _wrap_to_center(lon, center):
    return ((lon - center + 180.0) % 360.0) + center - 180.0


def _lon_delta(lon, lon0):
    return ((lon - lon0 + 180.0) % 360.0) - 180.0


def _box_mask(lon, lat, boxes):
    mask = np.zeros((lat.size, lon.size), dtype=bool)
    for _, west, east, south, north in boxes:
        if west <= east:
            lon_mask = (lon >= west) & (lon <= east)
        else:
            lon_mask = (lon >= west) | (lon <= east)
        lat_mask = (lat >= south) & (lat <= north)
        mask[np.ix_(lat_mask, lon_mask)] = True
    return mask


def _nearest_index(values, target):
    return int(np.nanargmin(np.abs(values - target)))


def _as_unwrapped_points(points):
    unwrapped = []
    previous = None
    for lon, lat in points:
        lon = float(lon)
        if previous is not None:
            while lon - previous > 180.0:
                lon -= 360.0
            while lon - previous < -180.0:
                lon += 360.0
        unwrapped.append((lon, float(lat)))
        previous = lon
    return unwrapped


def _densify_points(points, spacing_deg=0.12):
    points = _as_unwrapped_points(points)
    dense = []
    for (lon0, lat0), (lon1, lat1) in zip(points[:-1], points[1:]):
        mean_lat = 0.5 * (lat0 + lat1)
        segment_size = max(abs(lon1 - lon0) * np.cos(np.deg2rad(mean_lat)), abs(lat1 - lat0))
        n_steps = max(int(np.ceil(segment_size / spacing_deg)), 1)
        for t in np.linspace(0.0, 1.0, n_steps, endpoint=False):
            dense.append((lon0 + t * (lon1 - lon0), lat0 + t * (lat1 - lat0)))
    dense.append(points[-1])
    return dense


def _buffered_polyline_mask(lon, lat, points, width_deg):
    mask = np.zeros((lat.size, lon.size), dtype=bool)
    spacing = max(width_deg / 2.0, 0.08)
    for lon0, lat0 in _densify_points(points, spacing_deg=spacing):
        coslat = max(np.cos(np.deg2rad(lat0)), 0.2)
        lat_idx = np.where(np.abs(lat - lat0) <= width_deg)[0]
        lon_idx = np.where(np.abs(_lon_delta(lon, lon0)) <= width_deg / coslat)[0]
        if lat_idx.size == 0 or lon_idx.size == 0:
            continue
        yy = lat[lat_idx][:, None]
        xx = _lon_delta(lon[lon_idx][None, :], lon0) * coslat
        local_mask = (yy - lat0) ** 2 + xx ** 2 <= width_deg ** 2
        mask[np.ix_(lat_idx, lon_idx)] |= local_mask
    return mask


def _least_cost_route(z, lon, lat, start, end, depth, pad_deg):
    lon0, lat0 = start
    lon1, lat1 = end
    if abs(lon1 - lon0) > 180.0:
        return [start, end]

    lat_idx = np.where(
        (lat >= max(min(lat0, lat1) - pad_deg, float(lat.min())))
        & (lat <= min(max(lat0, lat1) + pad_deg, float(lat.max())))
    )[0]
    lon_idx = np.where(
        (lon >= max(min(lon0, lon1) - pad_deg, float(lon.min())))
        & (lon <= min(max(lon0, lon1) + pad_deg, float(lon.max())))
    )[0]

    z_sub = z[np.ix_(lat_idx, lon_idx)]
    z_fill = np.where(np.isfinite(z_sub), z_sub, 0.0)
    cost = 1.0 + np.clip(z_fill / depth, 0.0, 12.0) ** 2
    cost[(~np.isfinite(z_sub)) | (z_sub <= depth)] = 0.05

    route_start = (_nearest_index(lat[lat_idx], lat0), _nearest_index(lon[lon_idx], lon0))
    route_end = (_nearest_index(lat[lat_idx], lat1), _nearest_index(lon[lon_idx], lon1))
    route, _ = graph.route_through_array(cost, route_start, route_end, fully_connected=True, geometric=True)
    route = np.asarray(route)
    return list(zip(lon[lon_idx][route[:, 1]], lat[lat_idx][route[:, 0]]))


def _build_bridge_mask(z, lon, lat, depth, bridges):
    bridge_mask = np.zeros((lat.size, lon.size), dtype=bool)
    bridge_routes = {}

    for bridge in bridges:
        if bridge.get("method", "route") == "line":
            route = bridge["points"]
        else:
            route = []
            for start, end in zip(bridge["points"][:-1], bridge["points"][1:]):
                segment = _least_cost_route(z, lon, lat, start, end, depth, bridge.get("pad_deg", 4.0))
                if route:
                    segment = segment[1:]
                route.extend(segment)

        bridge_routes[bridge["name"]] = np.asarray(route, dtype=float)
        bridge_mask |= _buffered_polyline_mask(lon, lat, route, bridge.get("width_deg", 0.45))

    return bridge_mask, bridge_routes


def _large_barrier_mask(raw_barrier, min_cells):
    labels = measure.label(raw_barrier, connectivity=2)
    sizes = np.bincount(labels.ravel())
    keep = sizes >= min_cells
    keep[0] = False
    return keep[labels]


def make_working_bathymetry(
    deptho,
    depth,
    *,
    min_barrier_cells=2500,
    ignore_south_of=-60.0,
    north_cap_lat=72.0,
    bridges=BRIDGES,
    exclude_island_boxes=EXCLUDED_ISLAND_BOXES,
):
    """Return a bathymetry field suitable for single-valued basin slicing.

    Small land/shelf components are ignored by setting them deep. Narrow bridge
    masks are set shallow only after that cleanup, so they close specific gaps
    without broad rectangular dams.
    """
    deptho = deptho.transpose("latitude", "longitude").load()
    lat = deptho.latitude.values.astype(float)
    lon = deptho.longitude.values.astype(float)
    z = deptho.values.astype(np.float32, copy=False)

    finite = np.isfinite(z)
    raw_barrier = (~finite) | (z <= depth)
    exclude_island_mask = _box_mask(lon, lat, exclude_island_boxes)
    barrier_for_filter = raw_barrier.copy()
    barrier_for_filter[exclude_island_mask] = False
    if ignore_south_of is not None:
        barrier_for_filter[lat <= ignore_south_of, :] = False
    if north_cap_lat is not None:
        barrier_for_filter[lat >= north_cap_lat, :] = True

    large_barrier = _large_barrier_mask(barrier_for_filter, min_barrier_cells)
    bridge_mask, bridge_routes = _build_bridge_mask(z, lon, lat, depth, bridges)

    final_barrier = large_barrier | bridge_mask
    final_barrier[exclude_island_mask] = False
    if north_cap_lat is not None:
        final_barrier[lat >= north_cap_lat, :] = True

    z_work = z.copy()
    z_work[~finite] = 0.0
    z_work[raw_barrier & ~large_barrier & ~bridge_mask] = depth + 2000.0
    z_work[bridge_mask] = 0.0
    if north_cap_lat is not None:
        z_work[lat >= north_cap_lat, :] = 0.0

    return {
        "working_bathymetry": xr.DataArray(z_work, coords=deptho.coords, dims=deptho.dims),
        "raw_barrier_mask": xr.DataArray(raw_barrier, coords=deptho.coords, dims=deptho.dims),
        "large_barrier_mask": xr.DataArray(large_barrier, coords=deptho.coords, dims=deptho.dims),
        "bridge_mask": xr.DataArray(bridge_mask, coords=deptho.coords, dims=deptho.dims),
        "exclude_island_mask": xr.DataArray(exclude_island_mask, coords=deptho.coords, dims=deptho.dims),
        "barrier_mask": xr.DataArray(final_barrier, coords=deptho.coords, dims=deptho.dims),
        "bridge_routes": bridge_routes,
    }


def _boolean_runs(mask):
    padded = np.r_[False, mask, False]
    changes = np.diff(padded.astype(int))
    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0] - 1
    return list(zip(starts, ends))


def _crossing_or_fallback(x0, z0, x1, z1, depth, fallback):
    if np.isfinite(z0) and np.isfinite(z1) and z1 != z0 and (z0 - depth) * (z1 - depth) <= 0.0:
        x_cross = x0 + (depth - z0) * (x1 - x0) / (z1 - z0)
        if min(x0, x1) <= x_cross <= max(x0, x1):
            return float(x_cross), True
    return float(fallback), False


def _select_basin_run(x, deep, seed_lon):
    runs = _boolean_runs(deep)
    if not runs:
        return None

    for start, end in runs:
        if x[start] <= seed_lon <= x[end]:
            return start, end

    # If the seed is not wet at this latitude, choose the nearest substantial
    # interval, mildly favoring wider intervals over tiny straits/embayments.
    scores = []
    for start, end in runs:
        midpoint = 0.5 * (x[start] + x[end])
        width = x[end] - x[start]
        scores.append(abs(midpoint - seed_lon) - 0.25 * width)
    return runs[int(np.argmin(scores))]


def extract_basin_boundaries(deptho, depth=500.0, *, basin_configs=BASIN_CONFIGS, **working_kwargs):
    """Extract single-valued west/east longitude functions for each basin.

    Returns an xarray Dataset with dimensions (basin, latitude). Longitudes are
    stored in each basin's unwrapped coordinate, so Pacific values run roughly
    105..290 instead of jumping at the dateline.
    """
    deptho = deptho.transpose("latitude", "longitude").load()
    lat = deptho.latitude.values.astype(float)
    lon = deptho.longitude.values.astype(float)
    geometry = make_working_bathymetry(deptho, depth, **working_kwargs)
    z_work = geometry["working_bathymetry"].values

    basin_datasets = []
    for basin, config in basin_configs.items():
        lon_unwrapped = _wrap_to_center(lon, config["center"])
        order = np.argsort(lon_unwrapped)
        x = lon_unwrapped[order]
        z_sorted = z_work[:, order]
        domain = (x >= config["lon_bounds"][0]) & (x <= config["lon_bounds"][1])
        domain_idx = np.where(domain)[0]
        seed_lon = _wrap_to_center(config["seed_lon"], config["center"])

        lon_west = np.full(lat.size, np.nan)
        lon_east = np.full(lat.size, np.nan)
        west_is_shelf_crossing = np.zeros(lat.size, dtype=bool)
        east_is_shelf_crossing = np.zeros(lat.size, dtype=bool)

        for lat_i, latitude in enumerate(lat):
            if latitude < config["lat_bounds"][0] or latitude > config["lat_bounds"][1]:
                continue

            row = z_sorted[lat_i, domain_idx]
            deep = np.isfinite(row) & (row > depth)
            run = _select_basin_run(x[domain_idx], deep, seed_lon)
            if run is None:
                continue

            start, end = run
            left = domain_idx[start]
            right = domain_idx[end]
            if x[right] - x[left] < 2.0:
                continue

            if left > 0:
                lon_west[lat_i], west_is_shelf_crossing[lat_i] = _crossing_or_fallback(
                    x[left - 1], z_sorted[lat_i, left - 1], x[left], z_sorted[lat_i, left], depth, x[left]
                )
            else:
                lon_west[lat_i] = x[left]

            if right + 1 < x.size:
                lon_east[lat_i], east_is_shelf_crossing[lat_i] = _crossing_or_fallback(
                    x[right], z_sorted[lat_i, right], x[right + 1], z_sorted[lat_i, right + 1], depth, x[right]
                )
            else:
                lon_east[lat_i] = x[right]

        west_lat_mask = lat >= config["west_south"]
        east_lat_mask = lat >= config["east_south"]
        lon_west[~west_lat_mask] = np.nan
        lon_east[~east_lat_mask] = np.nan
        west_is_shelf_crossing[~west_lat_mask] = False
        east_is_shelf_crossing[~east_lat_mask] = False

        basin_datasets.append(
            xr.Dataset(
                {
                    "lon_west": ("latitude", lon_west),
                    "lon_east": ("latitude", lon_east),
                    "west_is_shelf_crossing": ("latitude", west_is_shelf_crossing),
                    "east_is_shelf_crossing": ("latitude", east_is_shelf_crossing),
                },
                coords={"latitude": lat},
                attrs={
                    "west_south": float(config["west_south"]),
                    "east_south": float(config["east_south"]),
                },
            ).assign_coords(basin=basin)
        )

    boundaries = xr.concat(basin_datasets, dim="basin")
    boundaries["basin_width_deg"] = boundaries.lon_east - boundaries.lon_west
    boundaries.attrs.update(
        {
            "depth_m": float(depth),
            "longitude_note": "Longitudes are basin-unwrapped; Pacific is approximately 0..360.",
            "single_valued_note": "At each latitude and basin, one deep interval is selected; west/east sides then receive independent schematic southern limits.",
        }
    )
    return boundaries, geometry


def _contiguous_slices(mask):
    padded = np.r_[False, mask, False]
    changes = np.diff(padded.astype(int))
    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0]
    return list(zip(starts, ends))


def _rolling_median(values, window_points):
    median = np.full_like(values, np.nan, dtype=float)
    half_window = window_points // 2
    for i in range(values.size):
        start = max(0, i - half_window)
        end = min(values.size, i + half_window + 1)
        local = values[start:end]
        local = local[np.isfinite(local)]
        if local.size >= 3:
            median[i] = np.median(local)
    return median


def _fill_small_gaps(latitude, values, max_gap_deg):
    filled = values.copy()
    valid = np.isfinite(filled)
    if valid.sum() < 2:
        return filled

    valid_idx = np.where(valid)[0]
    lo, hi = valid_idx[0], valid_idx[-1]
    y = latitude[lo : hi + 1]
    x = filled[lo : hi + 1]
    missing = ~np.isfinite(x)
    if not missing.any():
        return filled

    interpolated = np.interp(y, y[np.isfinite(x)], x[np.isfinite(x)])
    for start, end in _contiguous_slices(missing):
        gap_deg = y[end - 1] - y[start]
        has_left = start > 0 and np.isfinite(x[start - 1])
        has_right = end < x.size and np.isfinite(x[end])
        if has_left and has_right and gap_deg <= max_gap_deg:
            x[start:end] = interpolated[start:end]

    filled[lo : hi + 1] = x
    return filled


def _smooth_valid_segments(latitude, values, sigma_deg):
    smoothed = values.copy()
    dlat = float(np.nanmedian(np.diff(latitude)))
    sigma_points = max(sigma_deg / dlat, 0.1)

    for start, end in _contiguous_slices(np.isfinite(smoothed)):
        if end - start < 7:
            continue
        segment = smoothed[start:end]
        smooth_segment = ndimage.gaussian_filter1d(segment, sigma=sigma_points, mode="nearest")
        smooth_segment[0] = segment[0]
        smooth_segment[-1] = segment[-1]
        smoothed[start:end] = smooth_segment

    return smoothed


def _clean_single_boundary(
    latitude,
    longitude,
    *,
    lat_min,
    lat_max,
    rolling_window=205,
    outlier_threshold=6.0,
    max_step=7.0,
    max_gap_deg=4.0,
    smooth_sigma_deg=0.35,
):
    cleaned = longitude.astype(float).copy()
    cleaned[(latitude < lat_min) | (latitude > lat_max)] = np.nan

    for _ in range(3):
        rolling = _rolling_median(cleaned, rolling_window)
        outlier = (
            np.isfinite(cleaned)
            & np.isfinite(rolling)
            & (np.abs(cleaned - rolling) > outlier_threshold)
        )
        cleaned[outlier] = np.nan

        valid_idx = np.where(np.isfinite(cleaned))[0]
        if valid_idx.size < 2:
            continue
        jumps = np.abs(np.diff(cleaned[valid_idx])) > max_step
        for jump_i in np.where(jumps)[0]:
            left = valid_idx[jump_i]
            right = valid_idx[jump_i + 1]
            left_score = abs(cleaned[left] - rolling[left]) if np.isfinite(rolling[left]) else np.inf
            right_score = abs(cleaned[right] - rolling[right]) if np.isfinite(rolling[right]) else np.inf
            if left_score > right_score:
                cleaned[left] = np.nan
            elif right_score > left_score:
                cleaned[right] = np.nan
            else:
                cleaned[left] = np.nan
                cleaned[right] = np.nan

    cleaned = _fill_small_gaps(latitude, cleaned, max_gap_deg=max_gap_deg)
    cleaned = _smooth_valid_segments(latitude, cleaned, sigma_deg=smooth_sigma_deg)
    return cleaned


def clean_basin_boundary_functions(boundaries):
    """Clean six single-valued boundary functions without changing their meaning.

    This removes isolated side-sea/branch artifacts, fills small gaps left by
    outlier removal, and lightly smooths each valid latitude interval. Each of
    the six functions keeps its own schematic latitude domain.
    """
    cleaned_basins = []
    latitude = boundaries.latitude.values.astype(float)

    for basin in boundaries.basin.values:
        basin_name = str(basin)
        ds_basin = boundaries.sel(basin=basin).copy(deep=True)

        for side, variable, flag in [
            ("west", "lon_west", "west_is_shelf_crossing"),
            ("east", "lon_east", "east_is_shelf_crossing"),
        ]:
            params = {
                "lat_min": float(BOUNDARY_SOUTHERN_LIMITS[basin_name][side]),
                "lat_max": float(BOUNDARY_NORTHERN_LIMITS[basin_name][side]),
                "outlier_threshold": 6.0,
                "max_step": 7.0,
                "max_gap_deg": 4.0,
                "smooth_sigma_deg": 0.35,
            }
            if basin_name == "atlantic":
                params.update(outlier_threshold=5.0, max_step=5.0, smooth_sigma_deg=0.45)
                if side == "west":
                    params.update(max_gap_deg=6.0)
            elif basin_name == "indian":
                params.update(outlier_threshold=5.0, max_step=5.0, smooth_sigma_deg=0.45)
            elif basin_name == "pacific" and side == "west":
                params.update(outlier_threshold=5.0, max_step=5.0, max_gap_deg=5.0, smooth_sigma_deg=0.55)

            cleaned_values = _clean_single_boundary(latitude, ds_basin[variable].values, **params)
            ds_basin[variable].values[:] = cleaned_values
            ds_basin[flag].values[:] = ds_basin[flag].values & np.isfinite(cleaned_values)

        ds_basin["basin_width_deg"] = ds_basin.lon_east - ds_basin.lon_west
        cleaned_basins.append(ds_basin)

    cleaned = xr.concat(cleaned_basins, dim="basin")
    cleaned.attrs.update(boundaries.attrs)
    cleaned.attrs["cleaning_note"] = (
        "Boundary functions were robustly filtered, small gaps interpolated, "
        "and lightly smoothed after raw row-wise extraction."
    )
    return cleaned


def boundary_at_latitude(boundaries, basin, latitude):
    """Convenience interpolation for lon_west/lon_east at arbitrary latitude."""
    return boundaries.sel(basin=basin).interp(latitude=latitude)


In [4]:
DEPTHS = (500.0, 1000.0)
OUTPUT_PATHS = {
    500: ISOBATH_DIRECTORY / "global_isobath_GEBCO_500m.nc",
    1000: ISOBATH_DIRECTORY / "global_isobath_GEBCO_1000m.nc",
}


def _as_export_dataset(boundaries, depth):
    basin_lookup = {"P": "pacific", "A": "atlantic", "I": "indian"}

    def values(basin, variable):
        return boundaries.sel(basin=basin)[variable].values.astype(float)

    dataset = xr.Dataset(
        data_vars={
            "x_wP": ("latitude", values("pacific", "lon_west")),
            "x_wA": ("latitude", values("atlantic", "lon_west")),
            "x_wI": ("latitude", values("indian", "lon_west")),
            "x_eP": ("latitude", values("pacific", "lon_east")),
            "x_eA": ("latitude", values("atlantic", "lon_east")),
            "x_eI": ("latitude", values("indian", "lon_east")),
        },
        coords={"latitude": ("latitude", boundaries.latitude.values.astype(float))},
        attrs={
            "isobath_depth_m": float(depth),
            "source_dataset": "GEBCO 2026 sub-ice grid",
            "source_doi": "10.5285/4f68d5c7-45eb-f999-e063-7086abc036fa",
            "source_resolution_arc_seconds": 15.0,
            "working_resolution_arc_minutes": float(TARGET_RESOLUTION_ARCMIN),
            "longitude_note": "Longitudes are basin-unwrapped; Pacific is approximately 0..360.",
            "boundary_note": "Variables are six independent functions of latitude with NaN outside each function domain.",
            "source": "Generated by notebooks/extract_isobath_gebco.ipynb using the same extraction configuration as notebooks/extract_isobath.ipynb.",
        },
    )

    for suffix, basin in basin_lookup.items():
        dataset[f"x_w{suffix}"].attrs.update(
            long_name=f"western boundary longitude for {basin} basin",
            units="degrees_east",
            basin=basin,
            side="west",
        )
        dataset[f"x_e{suffix}"].attrs.update(
            long_name=f"eastern boundary longitude for {basin} basin",
            units="degrees_east",
            basin=basin,
            side="east",
        )
    return dataset


def _write_with_nan_fill(dataset, path):
    encoding = {name: {"_FillValue": np.nan} for name in dataset.data_vars}
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f"{path.stem}.tmp{path.suffix}")
    if temporary_path.exists():
        temporary_path.unlink()
    dataset.to_netcdf(temporary_path, encoding=encoding)
    temporary_path.replace(path)
    print(f"Wrote {path}")


gebco_products = {}
for depth in DEPTHS:
    print(f"\nExtracting {depth:g} m GEBCO isobaths...")
    boundaries_raw, geometry = extract_basin_boundaries(
        deptho,
        depth,
        min_barrier_cells=MIN_BARRIER_CELLS,
    )
    boundaries = clean_basin_boundary_functions(boundaries_raw)
    boundaries = boundaries.reindex(
        latitude=output_latitude,
        method="nearest",
        tolerance=TARGET_RESOLUTION_DEG / 10.0,
    )
    product = _as_export_dataset(boundaries, depth)
    _write_with_nan_fill(product, OUTPUT_PATHS[int(depth)])
    gebco_products[int(depth)] = product

    del boundaries_raw, boundaries, geometry
    gc.collect()

gebco_products


Extracting 500 m GEBCO isobaths...
Wrote /Users/awatford/Documents/GitHub/MOCAdjustmentTheory/data/tracked/isobath/global_isobath_GEBCO_500m.nc

Extracting 1000 m GEBCO isobaths...
Wrote /Users/awatford/Documents/GitHub/MOCAdjustmentTheory/data/tracked/isobath/global_isobath_GEBCO_1000m.nc


{500: <xarray.Dataset> Size: 571kB
 Dimensions:   (latitude: 10200)
 Coordinates:
   * latitude  (latitude) float64 82kB -79.99 -79.97 -79.96 ... 89.96 89.97 89.99
 Data variables:
     x_wP      (latitude) float64 82kB nan nan nan nan nan ... nan nan nan nan
     x_wA      (latitude) float64 82kB nan nan nan nan nan ... nan nan nan nan
     x_wI      (latitude) float64 82kB nan nan nan nan nan ... nan nan nan nan
     x_eP      (latitude) float64 82kB nan nan nan nan nan ... nan nan nan nan
     x_eA      (latitude) float64 82kB nan nan nan nan nan ... nan nan nan nan
     x_eI      (latitude) float64 82kB nan nan nan nan nan ... nan nan nan nan
 Attributes:
     isobath_depth_m:                 500.0
     source_dataset:                  GEBCO 2026 sub-ice grid
     source_doi:                      10.5285/4f68d5c7-45eb-f999-e063-7086abc0...
     source_resolution_arc_seconds:   15.0
     working_resolution_arc_minutes:  1.0
     longitude_note:                  Longitudes are basin-

In [5]:
for depth, product in gebco_products.items():
    print(f"\n{depth} m GEBCO product")
    for name in product.data_vars:
        valid = np.isfinite(product[name])
        print(
            f"  {name}: {int(valid.sum()):5d} points, "
            f"{float(product.latitude.where(valid).min()):6.2f} to "
            f"{float(product.latitude.where(valid).max()):6.2f} degrees latitude"
        )


500 m GEBCO product
  x_wP:  6206 points, -43.99 to  59.42 degrees latitude
  x_wA:  7202 points, -55.03 to  64.99 degrees latitude
  x_wI:  3587 points, -34.99 to  24.77 degrees latitude
  x_eP:  6926 points, -55.99 to  59.42 degrees latitude
  x_eA:  6000 points, -34.99 to  64.99 degrees latitude
  x_eI:  4132 points, -43.99 to  24.86 degrees latitude

1000 m GEBCO product
  x_wP:  6202 points, -43.99 to  59.36 degrees latitude
  x_wA:  7205 points, -55.07 to  64.99 degrees latitude
  x_wI:  3580 points, -34.99 to  24.66 degrees latitude
  x_eP:  6922 points, -55.99 to  59.36 degrees latitude
  x_eA:  6000 points, -34.99 to  64.99 degrees latitude
  x_eI:  4124 points, -43.99 to  24.72 degrees latitude
